[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-09-subword-tokenization.ipynb)

# Lecture 9: what your model cannot say, and subwords that can

Your A8 model has a **fixed vocabulary**: every word it saw in training, and nothing else. A
word it never saw becomes `<unk>`, and it can never produce that word in a translation, however
well it trains. This notebook measures how much of *your* test set that is, builds a subword
vocabulary that has no such gap, and then, in Part B, retrains your A8 model with it. That
retraining **is Assignment 9**; everything A9 needs happens in this one notebook, and nothing
sends you back to the A8 notebook.

**Part A, in class, on a CPU** (Steps 1 to 6). When it ends you will have, in
`MyDrive/CS479/assignment8`:

1. **A SentencePiece model trained on your A8 training split**, at a vocabulary size you chose:
   `spm.model` and `spm.vocab`.
2. **Two numbers side by side**, from a counting function you wrote: how many test words your
   word vocabulary cannot represent, and how many your subword vocabulary cannot. The second
   should be zero.
3. **A round-trip check**: text encoded to pieces and decoded back, unchanged.
4. **The settings Part B trains with**, measured on your data and saved to Drive, and your
   prediction of what they will do to your BLEU.

**Part B, outside class, on a GPU.** Retrains with the pieces, translates the same test set,
scores it, and re-scores your A8 model the same way, so the two numbers in your write-up were
produced by the same code. It takes as long as A8's training did.

**Running in Google Colab?** Run the next two cells. The first installs TorchLingo; the second
connects Google Drive, where your A8 files are. There is nothing to uncomment.


<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 9 (*Handling Morphology and Terminology in NMT*). A head start on **A9**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [ ]:
# Connect Google Drive, where the A8 kickoff notebook wrote your split.
from torchlingo.colab import setup

env = setup(drive=True)
A8_DIR = env.drive_dir / "CS479" / "assignment8"

missing = [n for n in ("train.tsv", "val.tsv", "test.tsv") if not (A8_DIR / n).exists()]
if missing:
    raise SystemExit(
        f"Not found in {A8_DIR}: {', '.join(missing)}. These are written by Step 5 of the A8 "
        "kickoff notebook. Run that far first, or point A8_DIR at wherever your split is."
    )

import pandas as pd
from torchlingo.config import Config
from torchlingo.data_processing import NMTDataset
from torchlingo.data_processing.vocab import CharVocab, SentencePieceVocab
from torchlingo.diagnostics import uniform_loss
from torchlingo.preprocessing import train_sentencepiece

test = pd.read_csv(A8_DIR / "test.tsv", sep="\t", dtype=str, keep_default_na=False)
print(f"Found your A8 split in {A8_DIR}; {len(test):,} test pairs.")

# A language written without spaces between words averages one or two "words" a line. For it,
# A8's NO_SPACES switch made the target vocabulary characters, so that is what this compares.
NO_SPACES = test.tgt.str.split().str.len().mean() < 3
UNITS = "character" if NO_SPACES else "word"
if NO_SPACES:
    print("Your language is written without spaces: your A8 model's target vocabulary was "
          "characters (NO_SPACES = True), so characters are what the pieces are compared with.")

## Step 1: what your word vocabulary cannot say

This rebuilds the exact vocabulary your A8 model uses (every word in the training split) and
asks how many **test** words fall outside it. Each one is an `<unk>`: on the source side the model
cannot read it, and on the target side it cannot write it, ever.

**If your language is written without spaces**, your A8 model's target side was characters, not
words, and that is what this rebuilds: a character seen fewer than twice in training is your
`<unk>`. Read "word" as "character" for the target side, here and below.

The function that counts them is yours to write, and it is four lines. `vocab.encode(sentence,
add_special_tokens=False)` gives one sentence's token ids; `vocab.unk_idx` is the id every unknown
word becomes. Return how many ids are `unk_idx` and how many ids there are in all. The cell
checks your function on a case whose answer is known before trusting it on your data.

**Before you run it, write down a guess** at the target-side percentage. There is no right
guess; there is a number you can be surprised by.

In [ ]:
if NO_SPACES:   # the same vocabulary A8 built: English in words, your language in characters
    words = NMTDataset(A8_DIR / "train.tsv", tgt_vocab=CharVocab(), config=Config())
    words.tgt_vocab.build_vocab(words.tgt_texts)
else:
    words = NMTDataset(A8_DIR / "train.tsv", config=Config())   # the same vocabulary A8 built

# Your guess, before running: the percentage of target-side test tokens the word vocabulary
# cannot represent. Inflected languages run higher. Any number; the point is to have one.
EXPECTED_UNK_PERCENT = None     # <- e.g. 5.0


def unknown_rate(vocab, sentences):
    """Return (unknown tokens, all tokens) for sentences encoded with vocab.

    TODO: write it. vocab.encode(s, add_special_tokens=False) gives the token ids of one
    sentence, and vocab.unk_idx is the id every unknown word becomes. Over all the sentences,
    count how many ids are unk_idx and how many ids there are in all.
    """
    raise NotImplementedError("write unknown_rate, then run this cell again")


# A case whose answer you know: one made-up word is one unknown token out of one.
assert unknown_rate(words.src_vocab, ["qzxvw"]) == (1, 1), "unknown_rate is not counting right yet"
assert EXPECTED_UNK_PERCENT is not None, "write your guess in EXPECTED_UNK_PERCENT first"

word_unk, word_total = {}, {}
for side, vocab in (("src", words.src_vocab), ("tgt", words.tgt_vocab)):
    unk, total = unknown_rate(vocab, test[side])
    word_unk[side], word_total[side] = unk, total
    units = UNITS if side == "tgt" else "word"
    print(f"{side}: {units} vocabulary {len(vocab):,};  test tokens it cannot represent: "
          f"{unk:,} of {total:,} ({100 * unk / total:.1f}%)")

print(f"\nYou guessed {EXPECTED_UNK_PERCENT:.1f}% on the target side.")
print(f"ln(V) for your target vocabulary: {uniform_loss(len(words.tgt_vocab)):.2f}")

The target-side percentage is the part to sit with. Those words are in your test references, so
every one of them is a word your model's translation is guaranteed to miss, and BLEU counts each
miss. No amount of training fixes it: the words are not in the vocabulary.

## Step 2: a subword vocabulary, trained on your training split only

SentencePiece learns a vocabulary of *pieces*: frequent words stay whole, and rare words are spelled
out of smaller pieces, down to single characters if need be. So any word made of characters it has
seen can be represented, including words it has never seen whole.

**Train it on `train.tsv` only.** Your test set must stay unseen, and a vocabulary is part of what a
model learns. Fitting it on everything would leak test material into the vocabulary, and nothing
would crash to tell you.

**The size is your choice.** 8,000 pieces is what the course model was measured with, and where
ln V is about 9.0, the number Lecture 7's slides quote. Fewer pieces (4,000): more words spelled
out, longer sequences, a smaller output layer. More pieces (16,000): more words kept whole,
shorter sequences, and more pieces seen too rarely to learn well. Pick one and say why in one
line; 8,000 is right if you have no reason to prefer another. Whichever you pick, A9 trains with
it, and the comparison is still one variable: your word vocabulary against this one. Training
takes about a minute.

In [ ]:
# Your choice. 8,000 is the course default; 4,000 and 16,000 are the alternatives worth a reason.
VOCAB_SIZE = 8_000
WHY = ""     # one line: why this size for your language

assert VOCAB_SIZE in (4_000, 8_000, 16_000), "pick one of the three sizes"
assert WHY.strip(), "say why, in one line, before training the vocabulary"

# A script with thousands of characters (Chinese, Japanese) cannot give every rare character a
# piece of its own at these sizes; 0.9995 lets the rarest fall back to <unk> instead of failing.
coverage = 0.9995 if NO_SPACES else None   # None: the library default, every character
train_sentencepiece([A8_DIR / "train.tsv"], model_prefix=str(A8_DIR / "spm"), vocab_size=VOCAB_SIZE,
                    character_coverage=coverage)
pieces = SentencePieceVocab(str(A8_DIR / "spm.model"))

print(f"Subword vocabulary: {len(pieces):,} pieces, shared by source and target.")
print(f"Why this size: {WHY.strip()}")
print(f"ln(V) = {uniform_loss(len(pieces)):.2f}")
print(f"Saved: {A8_DIR / 'spm.model'} and {A8_DIR / 'spm.vocab'}")

## Step 3: look at the pieces

`▁` marks the start of a word. A word that is one piece was common enough to earn its own entry;
a word split into several was rarer. Look for a word your word vocabulary called `<unk>`: here it
is spelled out of pieces instead of lost.

In [ ]:
label = UNITS.upper() + "S"
for sentence in test["tgt"].head(3):
    as_units = words.tgt_vocab.indices_to_tokens(words.tgt_vocab.encode(sentence, add_special_tokens=False))
    as_pieces = pieces.indices_to_tokens(pieces.encode(sentence, add_special_tokens=False))
    print(f"{label:10}", " ".join(as_units))
    print(f"{'PIECES':10}", " ".join(as_pieces))
    print()

## Step 4: the same count, with subwords

The same test sets, encoded with the pieces. The count should be zero. The only way it is not: a
*character* in the test set that never appears anywhere in training, such as a symbol or a letter
from another script. The cell names them if so.

In [ ]:
train = pd.read_csv(A8_DIR / "train.tsv", sep="\t", dtype=str, keep_default_na=False)

piece_unk = {}
for side in ("src", "tgt"):
    unk, total = unknown_rate(pieces, test[side])
    piece_unk[side] = unk
    units = UNITS if side == "tgt" else "word"
    print(f"{side}: test tokens the subword vocabulary cannot represent: {unk:,} of {total:,}   "
          f"({units} vocabulary: {word_unk[side]:,})")
    if unk:
        seen = set("".join(train["src"]) + "".join(train["tgt"]))
        unseen = sorted(set("".join(test[side])) - seen)
        print(f"   characters never seen in training: {unseen}")

## Step 5: nothing is lost on the way through

Pieces are only useful if they turn back into the same text. Ten test sentences, encoded and
decoded. One subtlety: SentencePiece normalizes text first (Unicode NFKC, and runs of spaces
collapse), so the check compares against the normalized sentence. A sentence that differs before
normalization differs only in how a character was typed, not in what it says.

In [ ]:
import unicodedata


def normalized(text):
    """What SentencePiece's default normalization does: NFKC, single spaces."""
    return " ".join(unicodedata.normalize("NFKC", text).split())


sample = list(test["tgt"].head(10))
round_trip = [pieces.decode(pieces.encode(s, add_special_tokens=False)) for s in sample]
exact = sum(a == b for a, b in zip(sample, round_trip))
after_normalizing = sum(normalized(a) == b for a, b in zip(sample, round_trip))

print(f"Round trip: {exact}/10 identical; {after_normalizing}/10 identical after normalizing.")
assert after_normalizing == 10, "a sentence changed beyond normalization; ask the teaching staff"
print(f"\nexample:\n  {sample[0]}\n  {' '.join(pieces.indices_to_tokens(pieces.encode(sample[0], add_special_tokens=False)))}\n  {round_trip[0]}")

## Step 6: the price, and the settings Part B trains with

Pieces are shorter than words, so every sentence gets **longer**, measured in tokens. That costs
memory and time, and it moves one setting you have never had to think about: the **longest
translation the model is allowed to produce**. TorchLingo stops decoding at 100 tokens by default.
Under A8's word vocabulary that matched the 100-word cap. In pieces, 100 tokens is well short of
100 words, so a long translation would be cut off mid-sentence, and your A9 BLEU would drop for a
reason that has nothing to do with subwords. The cell measures your own longest sentence and sets
the limit from it. (**Without spaces**, it runs the other way: a piece is often several characters,
so your sentences get *shorter*. The limit is measured either way.)

Then, with the two `<unk>` counts and the ratio in front of you, **predict A9**: will BLEU go
up or down against your A8 score, by roughly how much, and why? A prediction you wrote before
training is worth more than any explanation written after, and the A9 write-up asks for it back
with one sentence reconciling it with what happened.

The cell ends by saving everything Part B needs (your vocabulary size and reason, the decode
limit, the two `<unk>` counts, the ratio, your prediction) to `a9-settings.json` beside your
split. Part B runs in a fresh session with a GPU, and it reads that file rather than making you
run Steps 1 to 6 again.


In [ ]:
if NO_SPACES:
    unit_len = train["tgt"].str.replace(" ", "").str.len()     # A8's tokens were characters
else:
    unit_len = train["tgt"].str.split().str.len()
piece_len = train["tgt"].map(lambda s: len(pieces.encode(s, add_special_tokens=False)))

MAX_DECODE = int(piece_len.max()) + 10    # your longest target, in pieces, with room to spare
PIECE_RATIO = piece_len.sum() / unit_len.sum()

print(f"Target sentences are {PIECE_RATIO:.2f}x as many pieces as {UNITS}s.")
print(f"Longest training target: {unit_len.max()} {UNITS}s, {piece_len.max()} pieces.")
print(f"Part B will let the decoder write up to {MAX_DECODE} pieces (max_decode_length).")

# Your prediction for A9, in one or two sentences: BLEU up or down against A8, by roughly how
# much, and why. Use the two <unk> counts and the ratio above.
BLEU_PREDICTION = ""

assert BLEU_PREDICTION.strip(), "write your prediction before the settings are saved"

import json

settings = {
    "vocab_size": VOCAB_SIZE,
    "why": WHY.strip(),
    "no_spaces": bool(NO_SPACES),
    "max_decode_length": MAX_DECODE,
    "piece_ratio": round(float(PIECE_RATIO), 3),
    "unk_percent_words_tgt": round(100 * word_unk["tgt"] / word_total["tgt"], 2),
    "unk_count_pieces_tgt": int(piece_unk["tgt"]),
    "expected_unk_percent": float(EXPECTED_UNK_PERCENT),
    "bleu_prediction": BLEU_PREDICTION.strip(),
}
(A8_DIR / "a9-settings.json").write_text(json.dumps(settings, indent=2, ensure_ascii=False), encoding="utf-8")

print()
print("=== Saved for Part B:", A8_DIR / "a9-settings.json", "===")
for key, value in settings.items():
    print(f"  {key:24} {value}")
print()
print("Part B trains with these, under its own names (assignment-9, best-a9/, test-a9.hyp), so your")
print("A8 model and translations stay where they are. Nothing to paste anywhere.")


## What A9 is, now that you have seen this

Assignment 9 retrains your A8 model with the subword vocabulary and compares the two BLEU scores.
Because it uses **the same files and the same split**, the tokenizer is the only thing that
changes, so the difference in BLEU is the tokenizer's doing. That is the whole point of the
comparison, and why nothing here re-splits your data. **Part B below does it**, in this notebook.

Three rules Part B follows, so the comparison means something:

- **It scores the translations `translate_batch` returns.** They are ordinary text: the pieces
  are decoded back into words before anyone sees them. Token ids or pieces are not comparable
  with A8's word-level BLEU.
- **It decodes up to the limit Step 6 measured**, or long translations are cut short and the
  comparison is lost to truncation.
- **It keeps A8's results.** A8 saved its best model in `best/` and its translations in
  `test.hyp`, under the run named `assignment-8`. Part B uses `assignment-9`, `best-a9/` and
  `test-a9.hyp`, and it loads your A8 model from `best/` to score it again with the same code,
  so both numbers in your write-up come from one place.

The A9 write-up asks for: your A8 and A9 BLEU, scored the same way, and which BLEU; your
prediction from Step 6, and one sentence reconciling it with the result; the two target-side
`<unk>` rates and the piece-per-word ratio; your vocabulary size and why; one test sentence where
the two systems' outputs differ, with your opinion of which is better; and a paragraph on the
quality difference overall. Part B's last cell prints the numbers in that order.


---
## Report back

We will build a table on the board.

1. **Your target-side `<unk>` percentage, and your guess.** Who was furthest off, and which
   languages in the room run highest?
2. **Your vocabulary size, and why.** Anyone not at 8,000?
3. **Your piece-per-word ratio**, and what it does to `max_decode_length`.
4. **Your A9 prediction**, in one sentence. We will come back to these when A9 is in.

---
# Part B: Assignment 9, outside class

Everything from here needs a GPU and takes as long as your A8 training did, so it is not for the
classroom. It picks up from what Part A saved to Drive: the split, `spm.model`, and
`a9-settings.json`. Nothing from Steps 1 to 6 has to be run again.

**Before you start:** Runtime → Change runtime type → an A100, L4 or G4, not the free T4. That
restarts the session, so run the two setup cells at the top of the notebook again (install, then
Drive), skip Part A, and continue here. If the session drops during training, run the training
cell again; it resumes from the checkpoint it keeps on Drive.


In [ ]:
# Part B, cell 1: the GPU, Drive, and what Part A saved.
import json
import random

import pandas as pd
import torch
from torchlingo.colab import setup
from torchlingo.config import Config
from torchlingo.data_processing import create_dataloaders
from torchlingo.diagnostics import padding_report, uniform_loss
from torchlingo.inference import translate_batch
from torchlingo.models import SimpleTransformer
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer

env = setup(gpu=True, drive=True)       # stops with instructions if there is no GPU
device = env.device
A8_DIR = env.drive_dir / "CS479" / "assignment8"
BEST_DIR = A8_DIR / "best-a9"           # A9's best model; A8's best/ is left alone

settings = json.loads((A8_DIR / "a9-settings.json").read_text(encoding="utf-8"))
NO_SPACES = settings["no_spaces"]
MAX_DECODE = settings["max_decode_length"]
SPM = str(A8_DIR / "spm.model")
test = pd.read_csv(A8_DIR / "test.tsv", sep="\t", dtype=str, keep_default_na=False)

# The number of epochs you trained A8 for. The handout said about 35; if you changed it, match it
# here, so that the two runs differ in their tokenizer and nothing else.
EPOCHS = 35
SEED = 479                              # the kickoff's seed, for the bucketing shuffle

print(f"Split and settings found in {A8_DIR}: {len(test):,} test pairs, "
      f"{settings['vocab_size']:,} pieces, decode limit {MAX_DECODE}.")
print(f"Your prediction, from Step 6: {settings['bleu_prediction']}")


## Part B, step 1: train with the pieces

The same model as A8 (`d_model` 512, 8 heads, 6 + 6 layers, feed-forward 2048) and the same
training call. Two things differ, and both come from Part A: the data loaders encode with
`spm.model` instead of building word vocabularies, so one shared vocabulary of your chosen size
serves both languages; and `max_decode_length` is the limit Step 6 measured. The embedding tables
and the output layer are smaller than A8's, so the parameter count the cell prints is lower.

**Memory.** Sentences are longer in pieces and attention's cost grows with length squared. On an
A100 the batch size of 64 is fine; if the training cell stops with "CUDA out of memory", set
`batch_size=32` in the `Config` below and run both cells again.


In [ ]:
config = Config(
    d_model=512,
    n_heads=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    d_ff=2048,
    batch_size=64,
    num_workers=0,
    max_decode_length=MAX_DECODE,         # from Step 6: the longest target in pieces, plus ten
)

random.seed(SEED)
train_loader, val_loader, src_vocab, tgt_vocab = create_dataloaders(
    A8_DIR / "train.tsv",
    A8_DIR / "val.tsv",
    use_bucketing=True,
    num_workers=0,
    config=config,
    use_sentencepiece=True,
    sp_model_path=SPM,
    sp_tgt_model_path=SPM,                # the same file twice: one model serves both languages
)
print(padding_report(train_loader))

torch.manual_seed(SEED)
model = SimpleTransformer(src_vocab_size=len(src_vocab), tgt_vocab_size=len(tgt_vocab), config=config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Shared vocabulary {len(tgt_vocab):,} pieces;  ln(V) = {uniform_loss(len(tgt_vocab)):.2f}")
print(f"Model: {n_params:,} parameters   Batches per epoch: {len(train_loader):,}   (x {EPOCHS} epochs)")


In [ ]:
# Hours. Start it and leave it; if the session drops, run this cell again and it resumes.
checkpointer = TrainingCheckpointer(experiment_name="assignment-9")   # its own name, not A8's

result = train_model(
    model,
    train_loader,
    val_loader=val_loader,
    num_epochs=EPOCHS,
    device=device,
    config=config,
    gradient_clip=1.0,
    save_dir=BEST_DIR,
    log_every=50,
    checkpointer=checkpointer,
)

print(f"Epochs run: {len(result.train_losses)}")
print(f"Final train loss {result.train_losses[-1]:.4f}   final val loss {result.val_losses[-1]:.4f}")
print(f"Best checkpoint: {result.best_checkpoint}")


## Part B, step 2: translate the test set, and score both models the same way

Two things happen here, both on the same 2,000 test sentences, both greedy, both scored with the
same BLEU (spBLEU if your language is written without spaces, SacreBLEU's default otherwise).

First the A9 model, from `best-a9/`: its translations go to `test-a9.hyp`, with any `<unk>` left
in. Then **your A8 model again**, loaded from `best/`: the cell rebuilds A8's word (or character)
vocabularies from the training split, exactly as the A8 notebook did, loads the saved weights, and
translates and scores. That is the A8 number to report, whatever your A8 notebook printed at the
time: it was produced by the same code as the A9 number, which is what "scored the same way" means.

Both can run in a fresh session a day later: run the setup cells, Part B cell 1, and the `config`
cell, skip the training cell, and run this.


In [ ]:
import sacrebleu
from torchlingo.checkpoint import load_checkpoint

TOKENIZE = "flores200" if NO_SPACES else "13a"      # spBLEU without spaces, the default otherwise
sources, references = list(test.src), list(test.tgt)


def translate_all(m, s_vocab, t_vocab, cfg):
    m.eval()
    out = []
    for start in range(0, len(sources), 64):
        out.extend(translate_batch(m, sources[start:start + 64], s_vocab, t_vocab,
                                   decode_strategy="greedy", device=device, config=cfg))
    return out


# --- A9: the subword model, best by validation loss
ckpt = load_checkpoint(BEST_DIR / "model_best.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
hyp_a9 = translate_all(model, src_vocab, tgt_vocab, config)
bleu_a9 = sacrebleu.corpus_bleu(hyp_a9, [references], tokenize=TOKENIZE)
(A8_DIR / "test-a9.hyp").write_text("\n".join(hyp_a9) + "\n", encoding="utf-8")
print(f"A9 (pieces):  {bleu_a9}")

# --- A8 again, the same way: its vocabularies rebuilt as the A8 notebook built them, its saved weights
config_a8 = Config(d_model=512, n_heads=8, num_encoder_layers=6, num_decoder_layers=6, d_ff=2048,
                   batch_size=64, num_workers=0)
_, _, src_vocab_a8, tgt_vocab_a8 = create_dataloaders(
    A8_DIR / "train.tsv", A8_DIR / "val.tsv", use_bucketing=True, num_workers=0, config=config_a8,
    target_units="characters" if NO_SPACES else "words",
)
model_a8 = SimpleTransformer(src_vocab_size=len(src_vocab_a8), tgt_vocab_size=len(tgt_vocab_a8),
                             config=config_a8).to(device)
ckpt_a8 = load_checkpoint(A8_DIR / "best" / "model_best.pt", map_location=device)
model_a8.load_state_dict(ckpt_a8["model_state_dict"])
hyp_a8 = translate_all(model_a8, src_vocab_a8, tgt_vocab_a8, config_a8)
bleu_a8 = sacrebleu.corpus_bleu(hyp_a8, [references], tokenize=TOKENIZE)
print(f"A8 (words):   {bleu_a8}")


## Part B, step 3: the numbers for your write-up

In the order the write-up asks for them. The last block shows the first few test sentences where
the two systems disagree; pick one for the write-up and say which output is better and why.


In [ ]:
which = "spBLEU (flores200)" if NO_SPACES else "SacreBLEU, default tokenizer"
units = "character" if NO_SPACES else "word"

print("=== For the A9 write-up ===")
print(f"A8 BLEU: {bleu_a8.score:.2f}    A9 BLEU: {bleu_a9.score:.2f}    both greedy, both {which}")
print(f"Prediction, written before training: {settings['bleu_prediction']}")
print(f"   (your sentence reconciling it with the result goes in the write-up)")
print(f"Target-side <unk>, {units} vocabulary: {settings['unk_percent_words_tgt']:.2f}% of test tokens;"
      f"  with pieces: {settings['unk_count_pieces_tgt']} tokens")
print(f"Pieces per {units}: {settings['piece_ratio']:.2f};  decode limit used: {MAX_DECODE}")
print(f"Vocabulary size: {settings['vocab_size']:,}.  Why: {settings['why']}")
print()
print("=== Where the two systems differ (first five) ===")
shown = 0
for s, r, h8, h9 in zip(sources, references, hyp_a8, hyp_a9):
    if h8 != h9:
        print(f"\nSRC {s}\nREF {r}\nA8  {h8}\nA9  {h9}")
        shown += 1
        if shown == 5:
            break
print()
print(f"Files in {A8_DIR}: test-a9.hyp (A9's translations), spm.model and spm.vocab, and your split.")
print("Those, with your test set, are what the shared folder gets; the numbers above go to Learning Suite.")
